In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
#import pyspark.sql.functions import *

spark = (
    SparkSession.builder
        .appName("leitura e escrita")
        .master("local[*]")  
        #.master("spark://spark-master:7077")   
  
        .getOrCreate()
)

26/09/18 20:31:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_clientes = spark.read.csv("/opt/spark/storage/E-Commerce/customers.csv")
df_clientes.limit(5).show()

+----------+----+------+----------+---------------+
|       _c0| _c1|   _c2|       _c3|            _c4|
+----------+----+------+----------+---------------+
|CustomerID| Age|  City|SignupDate|CustomerSegment|
|    100001|22.0|Tehran|2023-09-11|        Regular|
|    100002|55.0|Tabriz|2024-01-16|            VIP|
|    100003|49.0| Karaj|2025-07-31|            New|
|    100004|39.0|Tehran|2023-04-23|            New|
+----------+----+------+----------+---------------+



In [4]:
schema_map = """
    CustomerID int,
    Age DOUBLE,
    City STRING,
    SignupDate DATE,
    CustomerSegment STRING
"""


df_cliente2 = spark.read.csv(
    '/opt/spark/storage/E-Commerce/customers.csv',
    header=True,
    sep=',',
    schema=schema_map
)
df_cliente2.show(5)

+----------+----+------+----------+---------------+
|CustomerID| Age|  City|SignupDate|CustomerSegment|
+----------+----+------+----------+---------------+
|    100001|22.0|Tehran|2023-09-11|        Regular|
|    100002|55.0|Tabriz|2024-01-16|            VIP|
|    100003|49.0| Karaj|2025-07-31|            New|
|    100004|39.0|Tehran|2023-04-23|            New|
|    100005|38.0|Tehran|2023-04-29|        Regular|
+----------+----+------+----------+---------------+
only showing top 5 rows



In [8]:
df_cliente2.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Age: double (nullable = true)
 |-- City: string (nullable = true)
 |-- SignupDate: date (nullable = true)
 |-- CustomerSegment: string (nullable = true)



In [14]:
df_cliente3 = (
    spark.read.format("csv")
    .load("/opt/spark/storage/E-Commerce/customers.csv")
)
df_cliente3.show(5)

+----------+----+------+----------+---------------+
|       _c0| _c1|   _c2|       _c3|            _c4|
+----------+----+------+----------+---------------+
|CustomerID| Age|  City|SignupDate|CustomerSegment|
|    100001|22.0|Tehran|2023-09-11|        Regular|
|    100002|55.0|Tabriz|2024-01-16|            VIP|
|    100003|49.0| Karaj|2025-07-31|            New|
|    100004|39.0|Tehran|2023-04-23|            New|
+----------+----+------+----------+---------------+
only showing top 5 rows



In [21]:
(
    df_cliente2.write.format("parquet")
    .mode("overwrite") #Se já existir um arquivo/pasta nesse caminho, substitua pelo resultado novo.
    .save("/opt/spark/storage/output/customers.parquet")
    )

"""
write → vai gravar o DataFrame.
format("orc") → vai salvar no formato ORC.
mode("overwrite") → se customers.orc já existir, substitui.
save(...) → define onde salvar.
"""

'\nwrite → vai gravar o DataFrame.\nformat("orc") → vai salvar no formato ORC.\nmode("overwrite") → se customers.orc já existir, substitui.\nsave(...) → define onde salvar.\n'

In [20]:
(
    df_cliente2.write.format("json")
    .mode("overwrite")
    .save("/opt/spark/storage/output/customers.json")
    )

In [17]:
(
    df_cliente2.write.format("csv")
    .mode("overwrite")
    .option("header", "true")
    .options(delimiter= ";")
    .save("/opt/spark/storage/output/customers.csv")
    )

In [18]:
(
    df_cliente2.write.format("orc")
    .mode("overwrite")
    .save("/opt/spark/storage/output/customers.orc")
    )

In [22]:
df_cliente4 = (
    spark.read.format("orc")
    .load("/opt/spark/storage/output/customers.orc")
)
df_cliente4.show(5)

+----------+----+------+----------+---------------+
|CustomerID| Age|  City|SignupDate|CustomerSegment|
+----------+----+------+----------+---------------+
|    100001|22.0|Tehran|2023-09-11|        Regular|
|    100002|55.0|Tabriz|2024-01-16|            VIP|
|    100003|49.0| Karaj|2025-07-31|            New|
|    100004|39.0|Tehran|2023-04-23|            New|
|    100005|38.0|Tehran|2023-04-29|        Regular|
+----------+----+------+----------+---------------+
only showing top 5 rows



# REPARTITION DE DADOS

In [ ]:
'''
Por que o Spark salva como part-00000-...json?
No Apache Spark, todo DataFrame é dividido em partições.
Cada partição gera um arquivo de saída.

O Spark:
Executa o job
Processa os dados em partições
Grava 1 arquivo por partição
Como seu DataFrame tem 1 partição, ele gerou:

Se tivesse 4 partições → seriam part-00000, part-00001, part-00002, part-00003.

_SUCCESS
O job terminou com sucesso
Todas as partições foram escritas corretamente
Ele não contém dados, é só um marcador de sucesso (muito usado em HDFS, S3, ADLS).

'''

'\nPor que o Spark salva como part-00000-...json?\nNo Apache Spark, todo DataFrame é dividido em partições.\nCada partição gera um arquivo de saída.\n\n\nO Spark:\nExecuta o job\nProcessa os dados em partições\nGrava 1 arquivo por partição\nComo seu DataFrame tem 1 partição, ele gerou:\n\nSe tivesse 4 partições → seriam part-00000, part-00001, part-00002, part-00003.\n\n_SUCCESS\nO job terminou com sucesso\nTodas as partições foram escritas corretamente\nEle não contém dados, é só um marcador de sucesso (muito usado em HDFS, S3, ADLS).\n\n'

In [23]:
(
    df_cliente2.repartition(4)
    .write.format("json")
    .save("/opt/spark/storage/output/customers_repartitioned.json")
)

# PARTITION

In [25]:
df_cliente2.show(5)

+----------+----+------+----------+---------------+
|CustomerID| Age|  City|SignupDate|CustomerSegment|
+----------+----+------+----------+---------------+
|    100001|22.0|Tehran|2023-09-11|        Regular|
|    100002|55.0|Tabriz|2024-01-16|            VIP|
|    100003|49.0| Karaj|2025-07-31|            New|
|    100004|39.0|Tehran|2023-04-23|            New|
|    100005|38.0|Tehran|2023-04-29|        Regular|
+----------+----+------+----------+---------------+
only showing top 5 rows



In [26]:
# salvar particionado por alguma coluna

(
    df_cliente2.write.format("parquet")
    .mode("overwrite")
    .partitionBy("CustomerSegment")
    .save("/opt/spark/storage/output/customers_repart_segment")
    )

In [6]:
clientes_vip = spark.read.format("parquet").load("/opt/spark/storage/output/customers_repart_segment/CustomerSegment=VIP")
clientes_vip.show(10)

+----------+----+-------+----------+
|CustomerID| Age|   City|SignupDate|
+----------+----+-------+----------+
|    100002|55.0| Tabriz|2024-01-16|
|    100043|51.0|Mashhad|2023-04-25|
|    100058|25.0| Shiraz|2024-12-24|
|    100060|50.0| Tehran|2024-10-05|
|    100072|27.0|Mashhad|2023-01-10|
|    100076|40.0| Tehran|2023-03-02|
|    100085|48.0|  Rasht|2023-05-07|
|    100087|22.0|    Qom|2023-03-10|
|    100102|27.0|Mashhad|2025-07-17|
|    100104|18.0| Tabriz|2025-07-15|
+----------+----+-------+----------+
only showing top 10 rows



In [8]:
clientes_geral_repart= (spark.read
                        .format("parquet")
                        .load("/opt/spark/storage/output/customers_repart_segment"))

clientes_geral_repart.show()

+----------+----+-------+----------+---------------+
|CustomerID| Age|   City|SignupDate|CustomerSegment|
+----------+----+-------+----------+---------------+
|    100001|22.0| Tehran|2023-09-11|        Regular|
|    100005|38.0| Tehran|2023-04-29|        Regular|
|    100006|59.0|Isfahan|2023-11-27|        Regular|
|    100007|22.0|  Karaj|2024-10-22|        Regular|
|    100008|51.0| Tabriz|2025-06-02|        Regular|
|    100009|27.0|  Karaj|2024-09-08|        Regular|
|    100011|NULL| Tehran|2025-08-11|        Regular|
|    100012|64.0| Shiraz|2023-07-07|        Regular|
|    100013|53.0| Tehran|2025-09-14|        Regular|
|    100014|54.0|  Karaj|2023-12-09|        Regular|
|    100017|NULL|Isfahan|2024-12-08|        Regular|
|    100018|24.0|Isfahan|2024-04-17|        Regular|
|    100019|58.0| Tabriz|2023-12-28|        Regular|
|    100020|39.0|Isfahan|2024-04-28|        Regular|
|    100021|42.0|  Rasht|2025-05-16|        Regular|
|    100022|35.0|  Ahvaz|2023-06-17|        Re

In [23]:
# particionar por mais de uma coluna
(
    df_cliente2.write.format("parquet")
    .mode("overwrite")
    .partitionBy("CustomerSegment", "Age")
    .save("/opt/spark/storage/output/customers_repart_segment_age")

)

In [24]:
clientes_idade_vip = spark.read.format("parquet").load("/opt/spark/storage/output/customers_repart_segment_age/CustomerSegment=VIP/Age=18.0")
clientes_idade_vip.show(10)

+----------+-------+----------+
|CustomerID|   City|SignupDate|
+----------+-------+----------+
|    100104| Tabriz|2025-07-15|
|    101624|Isfahan|2024-11-14|
|    102202|Mashhad|2025-01-29|
|    102646|   NULL|2023-08-08|
|    102975|  Karaj|2024-07-12|
|    103505|  Karaj|2023-12-09|
|    103674| Tehran|2024-07-09|
|    103725|Isfahan|2024-11-17|
|    103898|Mashhad|2023-09-07|
|    104055|  Ahvaz|2023-02-05|
+----------+-------+----------+
only showing top 10 rows



In [26]:
clientes_idade_vip_colunas = (spark.read
                              .format("parquet")
                              .option("basePath", "/opt/spark/storage/output/customers_repart_segment_age")
                              .load("/opt/spark/storage/output/customers_repart_segment_age/CustomerSegment=VIP/Age=18.0")
                              )
clientes_idade_vip_colunas.show()

+----------+-------+----------+---------------+----+
|CustomerID|   City|SignupDate|CustomerSegment| Age|
+----------+-------+----------+---------------+----+
|    100104| Tabriz|2025-07-15|            VIP|18.0|
|    101624|Isfahan|2024-11-14|            VIP|18.0|
|    102202|Mashhad|2025-01-29|            VIP|18.0|
|    102646|   NULL|2023-08-08|            VIP|18.0|
|    102975|  Karaj|2024-07-12|            VIP|18.0|
|    103505|  Karaj|2023-12-09|            VIP|18.0|
|    103674| Tehran|2024-07-09|            VIP|18.0|
|    103725|Isfahan|2024-11-17|            VIP|18.0|
|    103898|Mashhad|2023-09-07|            VIP|18.0|
|    104055|  Ahvaz|2023-02-05|            VIP|18.0|
|    104768| Tabriz|2025-05-29|            VIP|18.0|
|    104931| Tehran|2025-09-12|            VIP|18.0|
|    105760|  Rasht|2024-06-12|            VIP|18.0|
|    106414| Shiraz|2025-07-03|            VIP|18.0|
|    107064|    Qom|2025-09-22|            VIP|18.0|
|    107363| Tehran|2025-07-02|            VIP

In [29]:
# teste performance

teste1 = spark.read.format("parquet").load("/opt/spark/storage/output/customers_repart_segment_age/CustomerSegment=VIP/Age=58.0")
teste1.show()

+----------+-------+----------+
|CustomerID|   City|SignupDate|
+----------+-------+----------+
|    100477| Tehran|2024-02-09|
|    100816|    Qom|2023-07-06|
|    101097| Tehran|2025-10-02|
|    101125| Tehran|2023-02-26|
|    101158| Tehran|2024-09-29|
|    102168| Tehran|2025-08-01|
|    102512| Shiraz|2023-06-04|
|    103190|    Qom|2024-07-15|
|    103332| Tabriz|2023-05-07|
|    104762| Tehran|2025-09-12|
|    104973| Kerman|2023-10-07|
|    105322| Shiraz|2024-01-22|
|    105935|Mashhad|2023-12-31|
|    106274| Tehran|2023-11-29|
|    106801|  Karaj|2025-08-10|
|    107640|Isfahan|2025-05-25|
|    109151| Tehran|2023-04-26|
|    109299| Tehran|2025-11-17|
|    109433|    Qom|2024-12-14|
|    109598|   NULL|2024-10-19|
+----------+-------+----------+
only showing top 20 rows



In [30]:
teste2 = spark.read.format("parquet").load("/opt/spark/storage/output/customers_repart_segment_age")
teste2.filter((F.col("CustomerSegment") == "VIP") & (F.col("Age") == 58)).show()

+----------+-------+----------+---------------+----+
|CustomerID|   City|SignupDate|CustomerSegment| Age|
+----------+-------+----------+---------------+----+
|    100477| Tehran|2024-02-09|            VIP|58.0|
|    100816|    Qom|2023-07-06|            VIP|58.0|
|    101097| Tehran|2025-10-02|            VIP|58.0|
|    101125| Tehran|2023-02-26|            VIP|58.0|
|    101158| Tehran|2024-09-29|            VIP|58.0|
|    102168| Tehran|2025-08-01|            VIP|58.0|
|    102512| Shiraz|2023-06-04|            VIP|58.0|
|    103190|    Qom|2024-07-15|            VIP|58.0|
|    103332| Tabriz|2023-05-07|            VIP|58.0|
|    104762| Tehran|2025-09-12|            VIP|58.0|
|    104973| Kerman|2023-10-07|            VIP|58.0|
|    105322| Shiraz|2024-01-22|            VIP|58.0|
|    105935|Mashhad|2023-12-31|            VIP|58.0|
|    106274| Tehran|2023-11-29|            VIP|58.0|
|    106801|  Karaj|2025-08-10|            VIP|58.0|
|    107640|Isfahan|2025-05-25|            VIP

### Outros tipos de leitura

In [34]:
clientes_csv_read = (
    spark.read.format("csv")
    .option("header", "true")
    .option("delimiter", ";")
    .load("/opt/spark/storage/output/customers.csv")
)

clientes_csv_read.show(5)

+----------+----+------+----------+---------------+
|CustomerID| Age|  City|SignupDate|CustomerSegment|
+----------+----+------+----------+---------------+
|    100001|22.0|Tehran|2023-09-11|        Regular|
|    100002|55.0|Tabriz|2024-01-16|            VIP|
|    100003|49.0| Karaj|2025-07-31|            New|
|    100004|39.0|Tehran|2023-04-23|            New|
|    100005|38.0|Tehran|2023-04-29|        Regular|
+----------+----+------+----------+---------------+
only showing top 5 rows



In [35]:
clientes_orc_read = (
    spark.read.format("orc")
    .load("/opt/spark/storage/output/customers.orc")
)

clientes_csv_read.show(5)

+----------+----+------+----------+---------------+
|CustomerID| Age|  City|SignupDate|CustomerSegment|
+----------+----+------+----------+---------------+
|    100001|22.0|Tehran|2023-09-11|        Regular|
|    100002|55.0|Tabriz|2024-01-16|            VIP|
|    100003|49.0| Karaj|2025-07-31|            New|
|    100004|39.0|Tehran|2023-04-23|            New|
|    100005|38.0|Tehran|2023-04-29|        Regular|
+----------+----+------+----------+---------------+
only showing top 5 rows



In [36]:
clientes_json_read = (
    spark.read.format("json")
    .load("/opt/spark/storage/output/customers.json")
)

clientes_csv_read.show(5)

+----------+----+------+----------+---------------+
|CustomerID| Age|  City|SignupDate|CustomerSegment|
+----------+----+------+----------+---------------+
|    100001|22.0|Tehran|2023-09-11|        Regular|
|    100002|55.0|Tabriz|2024-01-16|            VIP|
|    100003|49.0| Karaj|2025-07-31|            New|
|    100004|39.0|Tehran|2023-04-23|            New|
|    100005|38.0|Tehran|2023-04-29|        Regular|
+----------+----+------+----------+---------------+
only showing top 5 rows

